# Notebook 14 — Una LSTM che genera titoli di notizie in italiano

**Modulo:** 14 — Come funzionano davvero gli LLM
**Tempo stimato:** 1,5-2 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Un piccolo dataset di titoli di notizie in italiano, scritto per l'occasione
- Una pipeline completa di preprocessing: pulizia, tokenizzazione, n-grammi, padding
- Un modello LSTM minimale (Embedding -> LSTM -> Dropout -> Dense softmax)
- Una funzione di generazione di testo parola-per-parola (decodifica greedy)
- Un confronto tra decodifica greedy e campionamento con temperatura

## Prerequisiti

```bash
pip install tensorflow numpy
```

> **Nota:** a differenza dei notebook precedenti, qui **non serve nessuna chiave API**: il modello
> si addestra interamente in locale, su un dataset piccolissimo, in pochi minuti su CPU. L'obiettivo
> e l'intuizione del meccanismo, non la qualita del testo generato: con un dataset di poche decine
> di titoli, non aspettarti risultati sorprendenti.

---
## Cella 0 — Setup

Importiamo le librerie e fissiamo un seed casuale per rendere i risultati piu riproducibili tra
un'esecuzione e l'altra (non perfettamente identici, per via del comportamento interno di
TensorFlow, ma vicini).

In [ ]:
import numpy as np
import tensorflow as tf
import string
import unicodedata

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)


---
## Cella 1 — Il dataset: titoli di notizie in italiano

Un dataset volutamente piccolo (poche decine di titoli), scritto per questo notebook, che copre
alcuni temi ricorrenti (tecnologia, intelligenza artificiale, economia, clima, sport). L'obiettivo
non e addestrare un modello utile in produzione, ma vedere in azione, su scala ridotta, lo stesso
meccanismo che sta alla base di un LLM.

In [ ]:
titoli = [
    "l'intelligenza artificiale trasforma il mercato del lavoro",
    "il governo approva la nuova legge sulla privacy digitale",
    "la banca centrale alza i tassi di interesse per frenare l'inflazione",
    "un nuovo attacco informatico colpisce una grande azienda europea",
    "la squadra vince il campionato dopo dieci anni di attesa",
    "il cambiamento climatico minaccia le coste italiane",
    "le aziende tecnologiche investono miliardi nell'intelligenza artificiale",
    "il prezzo dell'energia scende dopo mesi di rialzi continui",
    "una startup italiana sviluppa un nuovo modello linguistico",
    "il ministero annuncia fondi per la ricerca scientifica",
    "aumentano gli investimenti nelle energie rinnovabili in europa",
    "il mercato azionario chiude in rialzo dopo i dati sull'occupazione",
    "gli scienziati scoprono un nuovo metodo per il riciclo della plastica",
    "la citta lancia un piano per la mobilita sostenibile",
    "cresce la domanda di competenze digitali nel mercato del lavoro",
    "un'universita italiana presenta un robot per l'assistenza domestica",
    "il governo discute una riforma della sanita pubblica",
    "la squadra nazionale si prepara per i prossimi campionati mondiali",
    "aumenta il numero di aziende che adottano l'intelligenza artificiale",
    "gli esperti avvertono sui rischi della disinformazione online",
    "una nuova legge regola l'uso dei dati personali nelle aziende",
    "il turismo digitale cresce grazie alle nuove piattaforme online",
    "la ricerca sul clima riceve nuovi finanziamenti pubblici",
    "un'azienda tecnologica presenta un nuovo chip per l'intelligenza artificiale",
    "la citta investe nella sicurezza informatica delle infrastrutture pubbliche",
    "cresce l'attenzione verso l'etica nell'intelligenza artificiale",
    "il settore automobilistico punta sui veicoli elettrici",
    "una nuova piattaforma aiuta le piccole imprese a digitalizzarsi",
    "gli investitori guardano con interesse al mercato dell'energia solare",
    "il governo presenta un piano nazionale per la cybersicurezza",
    "la squadra di ricerca pubblica uno studio sul cambiamento climatico",
    "aumentano le assunzioni nel settore dell'intelligenza artificiale",
    "una nuova app promette di semplificare la gestione delle finanze personali",
    "il parlamento discute una legge sulla sicurezza informatica",
    "gli agricoltori adottano sensori intelligenti per monitorare i campi",
    "la banca centrale pubblica nuove previsioni sull'economia nazionale",
    "cresce l'uso dei modelli linguistici nelle piccole aziende",
    "una ricerca universitaria analizza l'impatto dei social media sui giovani",
    "il governo lancia un programma di formazione sull'intelligenza artificiale",
    "la citta apre un nuovo centro per l'innovazione tecnologica",
    "gli scienziati sviluppano una nuova batteria a lunga durata",
    "cresce il numero di startup che lavorano sull'intelligenza artificiale",
    "una nuova normativa europea regola i modelli di intelligenza artificiale",
    "il settore energetico affronta una fase di forte trasformazione",
    "la ricerca scientifica italiana ottiene nuovi riconoscimenti internazionali",
    "aumentano gli attacchi informatici contro le piccole e medie imprese",
    "il governo stanzia nuovi fondi per la transizione digitale",
    "una startup presenta un assistente virtuale per la sanita pubblica",
    "cresce l'interesse degli investitori per l'intelligenza artificiale generativa",
    "la citta pianifica nuovi investimenti nella mobilita elettrica",
]

print(f"Numero di titoli nel dataset: {len(titoli)}")
print("\nPrimi 5 titoli:")
for t in titoli[:5]:
    print(" -", t)


---
## Cella 2 — Pulizia e normalizzazione del testo

Come descritto nella dispensa (Sezione 5.1): rimuoviamo la punteggiatura, gli accenti, e
convertiamo tutto in minuscolo. Questo riduce il rumore che il modello dovrebbe altrimenti
imparare a ignorare da solo.

In [ ]:
def preprocessa_testo(testo: str) -> str:
    """Rimuove punteggiatura, accenti, e converte in minuscolo."""
    testo = "".join(c for c in testo if c not in string.punctuation).lower()
    testo = unicodedata.normalize("NFKD", testo).encode("ascii", "ignore").decode("utf-8", "ignore")
    return testo


dataset_pulito = [preprocessa_testo(t) for t in titoli]

print("Prima:", titoli[0])
print("Dopo: ", dataset_pulito[0])


---
## Cella 3 — Tokenizzazione

Costruiamo il dizionario (vocabolario) del dataset con il `Tokenizer` di Keras, che assegna un
indice numerico a ogni parola unica del corpus (vedi dispensa, Sezione 5.2).

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

tokenizer = Tokenizer()
tokenizer.fit_on_texts(dataset_pulito)

totale_parole = len(tokenizer.word_index) + 1  # +1 per il token di padding (indice 0)

print(f"Parole uniche nel vocabolario: {totale_parole - 1}")
print(f"Dimensione totale del vocabolario (incluso padding): {totale_parole}")
print("\nEsempio di indici assegnati:")
for parola, indice in list(tokenizer.word_index.items())[:10]:
    print(f"  '{parola}' -> {indice}")


---
## Cella 4 — Coppie entrata/uscita con n-grammi

Per ogni titolo tokenizzato, generiamo tutte le sotto-sequenze crescenti: dalla coppia di 2 token
fino al titolo intero. Questo produce molti piu esempi di addestramento dei 50 titoli di partenza
(vedi dispensa, Sezione 5.3, e la domanda 4 di autovalutazione).

In [ ]:
def genera_sequenze_ngrammi(tokenizer, dataset_testi):
    """Trasforma ogni testo in una lista di sotto-sequenze crescenti di token."""
    sequenze = []
    for testo in dataset_testi:
        token_list = tokenizer.texts_to_sequences([testo])[0]
        for i in range(1, len(token_list)):
            n_gramma = token_list[:i + 1]
            sequenze.append(n_gramma)
    return sequenze


sequenze_ngrammi = genera_sequenze_ngrammi(tokenizer, dataset_pulito)

print(f"Titoli di partenza: {len(dataset_pulito)}")
print(f"Coppie entrata/uscita generate: {len(sequenze_ngrammi)}")
print("\nEsempio (i numeri sono indici del vocabolario, non parole):")
for seq in sequenze_ngrammi[:5]:
    print(" ", seq)


---
## Cella 5 — Padding e separazione X / y

Le sequenze hanno lunghezze diverse: le uniformiamo con padding a sinistra fino alla lunghezza
massima presente nel dataset. Poi separiamo l'ultimo token di ogni sequenza (l'etichetta `y`) dal
resto (l'input `X`), e trasformiamo `y` in formato one-hot per la classificazione multiclasse (vedi
dispensa, Sezioni 5.4 e 5.5).

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def prepara_dati_addestramento(sequenze_ngrammi, totale_parole):
    lunghezza_massima = max(len(seq) for seq in sequenze_ngrammi)
    sequenze_padded = np.array(
        pad_sequences(sequenze_ngrammi, maxlen=lunghezza_massima, padding="pre")
    )
    X = sequenze_padded[:, :-1]
    y = sequenze_padded[:, -1]
    y = to_categorical(y, num_classes=totale_parole)
    return X, y, lunghezza_massima


X_train, y_train, lunghezza_massima_sequenza = prepara_dati_addestramento(sequenze_ngrammi, totale_parole)

print(f"Forma di X_train: {X_train.shape}")
print(f"Forma di y_train: {y_train.shape}")
print(f"Lunghezza massima di sequenza (con padding): {lunghezza_massima_sequenza}")


---
## Cella 6 — Costruzione del modello LSTM

Architettura minimale, come descritto nella dispensa (Sezione 6):

1. **Embedding**: trasforma ogni indice di token in un vettore denso.
2. **LSTM**: modella le dipendenze nella sequenza.
3. **Dropout**: regolarizzazione, per ridurre l'overfitting su un dataset cosi piccolo.
4. **Dense + softmax**: distribuzione di probabilita su tutto il vocabolario.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout

def crea_modello(lunghezza_massima_sequenza: int, totale_parole: int) -> Sequential:
    lunghezza_input = lunghezza_massima_sequenza - 1

    modello = Sequential([
        Embedding(totale_parole, 10, input_length=lunghezza_input),
        LSTM(100),
        Dropout(0.1),
        Dense(totale_parole, activation="softmax"),
    ])

    modello.compile(loss="categorical_crossentropy", optimizer="adam")
    return modello


modello = crea_modello(lunghezza_massima_sequenza, totale_parole)
modello.summary()


---
## Cella 7 — Addestramento

Con un dataset cosi piccolo e un'architettura cosi semplice, l'addestramento richiede pochi minuti
su CPU. Non aspettarti una perdita (`loss`) bassissima: il dataset e minuscolo per gli standard di
un vero modello linguistico, e questo e voluto (vedi dispensa, Sezione 6).

In [ ]:
cronologia = modello.fit(X_train, y_train, epochs=80, verbose=2)


---
## Cella 8 — Generazione di testo: decodifica greedy

Implementiamo il ciclo di generazione descritto nella dispensa (Sezione 7): preprocessare il
prompt esattamente come il dataset, predire il vettore di probabilita, scegliere l'indice piu
probabile (argmax), concatenare la parola e ripetere.

In [ ]:
def genera_testo_greedy(prompt: str, numero_parole: int, modello, tokenizer, lunghezza_massima_sequenza: int) -> str:
    """Genera testo aggiungendo, a ogni passo, la parola con probabilita piu alta (argmax)."""
    testo_corrente = prompt
    for _ in range(numero_parole):
        # 1. Preprocessa il prompt corrente come abbiamo fatto con il dataset di addestramento
        prompt_pulito = preprocessa_testo(testo_corrente)
        sequenza = tokenizer.texts_to_sequences([prompt_pulito])[0]
        sequenza_padded = pad_sequences([sequenza], maxlen=lunghezza_massima_sequenza - 1, padding="pre")

        # 2. Predici la distribuzione di probabilita sul vocabolario
        probabilita = modello.predict(sequenza_padded, verbose=0)
        indice_predetto = np.argmax(probabilita, axis=1)[0]

        # 3. Converti l'indice nella parola corrispondente
        parola_predetta = ""
        for parola, indice in tokenizer.word_index.items():
            if indice == indice_predetto:
                parola_predetta = parola
                break

        # 4. Concatena la parola generata al testo
        testo_corrente += " " + parola_predetta

    return testo_corrente.title()


print(genera_testo_greedy("la ciberSicurezza", 6, modello, tokenizer, lunghezza_massima_sequenza))
print(genera_testo_greedy("l'intelligenza artificiale", 5, modello, tokenizer, lunghezza_massima_sequenza))
print(genera_testo_greedy("il governo", 6, modello, tokenizer, lunghezza_massima_sequenza))
print(genera_testo_greedy("la squadra", 5, modello, tokenizer, lunghezza_massima_sequenza))


**Le tue osservazioni** (modifica questa cella):

- I titoli generati sono coerenti a livello grammaticale? ...
- Riconosci pattern che vengono dal dataset di addestramento? ...
- Cosa succede se cambi il prompt di partenza con parole rare nel dataset? ...

---
## Esercizio — `# TODO`: campionamento con temperatura invece di argmax

Nella dispensa (Sezione 7.3) e nel Modulo 09 abbiamo visto che scegliere sempre la parola piu
probabile (decodifica *greedy*) e la strategia piu semplice, ma non l'unica. Un'alternativa comune
e il **campionamento con temperatura**: invece di prendere sempre l'argmax, si campiona una parola
dalla distribuzione di probabilita, dopo averla "ammorbidita" o "irrigidita" con un parametro di
temperatura (stesso concetto di `temperature` visto per le API di testo).

Completa la funzione qui sotto:
1. Applica la temperatura ai logaritmi delle probabilita: `probabilita ** (1 / temperatura)`.
2. Rinormalizza il risultato (deve sommare a 1).
3. Usa `np.random.choice` per campionare un indice secondo questa nuova distribuzione, invece di
   usare `argmax`.

In [ ]:
def genera_testo_con_temperatura(prompt: str, numero_parole: int, modello, tokenizer, lunghezza_massima_sequenza: int, temperatura: float = 0.8) -> str:
    """Come genera_testo_greedy, ma campiona la parola successiva invece di prendere sempre l'argmax."""
    testo_corrente = prompt
    for _ in range(numero_parole):
        prompt_pulito = preprocessa_testo(testo_corrente)
        sequenza = tokenizer.texts_to_sequences([prompt_pulito])[0]
        sequenza_padded = pad_sequences([sequenza], maxlen=lunghezza_massima_sequenza - 1, padding="pre")

        probabilita = modello.predict(sequenza_padded, verbose=0)[0]

        # TODO: applica la temperatura e rinormalizza
        # probabilita_scalate = probabilita ** (1.0 / temperatura)
        # probabilita_scalate = probabilita_scalate / np.sum(probabilita_scalate)
        probabilita_scalate = probabilita  # sostituisci con la versione scalata

        # TODO: campiona un indice secondo probabilita_scalate invece di usare argmax
        # indice_predetto = np.random.choice(len(probabilita_scalate), p=probabilita_scalate)
        indice_predetto = np.argmax(probabilita_scalate)  # sostituisci con il campionamento

        parola_predetta = ""
        for parola, indice in tokenizer.word_index.items():
            if indice == indice_predetto:
                parola_predetta = parola
                break

        testo_corrente += " " + parola_predetta

    return testo_corrente.title()


# Prova la tua implementazione (decommenta dopo aver completato i TODO)
# for temp in [0.3, 0.8, 1.3]:
#     print(f"temperatura={temp}: {genera_testo_con_temperatura('il governo', 6, modello, tokenizer, lunghezza_massima_sequenza, temperatura=temp)}")


---
## Sfida finale — Ampliare il dataset e confrontare i risultati

Il dataset che abbiamo usato e volutamente minuscolo. In questa sfida, mettiti nei panni di chi
deve migliorare un po' la qualita del modello senza cambiarne l'architettura.

1. Aggiungi almeno **15 nuovi titoli** in italiano alla lista `titoli` della Cella 1 (puoi copiarla
   qui sotto ed estenderla, oppure tornare su e modificarla direttamente).
2. Rilancia l'intera pipeline (pulizia -> tokenizzazione -> n-grammi -> padding -> training) sul
   dataset esteso.
3. Confronta la generazione di testo, con gli stessi prompt usati nella Cella 8, prima e dopo
   l'ampliamento del dataset.
4. Rispondi per iscritto: il numero di parole nel vocabolario e cambiato di molto? La qualita
   percepita dei titoli generati e migliorata? Cosa ti aspetteresti aggiungendo 500 titoli invece
   di 15 (collega la risposta alla Sezione 8 della dispensa, sul rapporto tra scala e capacita)?

In [ ]:
# Spazio per la sfida finale: estendi il dataset, riesegui la pipeline, confronta i risultati.
titoli_estesi = titoli + [
    # TODO: aggiungi almeno 15 nuovi titoli in italiano qui
]

# TODO: ripeti pulizia, tokenizzazione, n-grammi, padding, training sul dataset esteso
# (puoi riusare le funzioni definite sopra: preprocessa_testo, genera_sequenze_ngrammi,
#  prepara_dati_addestramento, crea_modello)


---
## Uno sguardo avanti: dove finisce la LSTM, dove inizia il Transformer

Il modello che hai appena addestrato soffre esattamente dei due limiti descritti nella dispensa
(Sezione 8): elabora la sequenza un token alla volta, e la sua "memoria" delle parole lontane nel
contesto e debole, anche con il meccanismo LSTM. Con un dataset e un'architettura di questa scala,
il problema e appena percepibile — ma diventa determinante quando si passa a sequenze lunghe e
modelli con miliardi di parametri.

Nel **Modulo 15** vediamo come il Transformer risolve entrambi i problemi con un unico meccanismo,
l'attenzione, che permette a ogni token di "guardare" direttamente qualunque altro token della
sequenza, in parallelo.

---
## Autovalutazione della settimana

Rispondi in questa cella prima di chiudere il notebook:

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la prossima settimana?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 14 — Modulo 14 del Corso Completo di IA Generativa*